#Assignment: Adult Census Income Dataset Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

## Task 1: Dataset Understanding

In [ ]:
df = pd.read_csv('/content/adult.csv')
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

print("Dataset loaded and cleaned. Column names:", df.columns.tolist())
display(df.head())

Dataset loaded and cleaned. Column names: ['age', 'workclass', 'fnlwgt', 'education', 'education.num', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'capital.gain', 'capital.loss', 'hours.per.week', 'native.country', 'income']


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [ ]:
print(f"Dataset Shape: {df.shape}")
print("\nTarget Variable Distribution:")
print(df['income'].value_counts(normalize=True))

Dataset Shape: (32561, 15)

Target Variable Distribution:
income
<=50K    0.75919
>50K     0.24081
Name: proportion, dtype: float64


## Task 1: Dataset Understanding

In [ ]:
df = pd.read_csv('/content/adult.csv')
# Clean column names and object values
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

print("Dataset loaded and cleaned. Column names:", df.columns.tolist())
display(df.head())

Dataset loaded and cleaned. Column names: ['age', 'workclass', 'fnlwgt', 'education', 'education.num', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'capital.gain', 'capital.loss', 'hours.per.week', 'native.country', 'income']


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [ ]:
print(f"Dataset Shape: {df.shape}")
print("\nTarget Variable Distribution:")
print(df['income'].value_counts(normalize=True))

Dataset Shape: (32561, 15)

Target Variable Distribution:
income
<=50K    0.75919
>50K     0.24081
Name: proportion, dtype: float64


## Task 2: Data Cleaning

### 2.1 Check for Missing Values

In [ ]:
print("Missing values before cleaning:")
display(df.isnull().sum()[df.isnull().sum() > 0])

Missing values before cleaning:


,0


### 2.2 Replace '?' with NaN

In [ ]:
df.replace('?', np.nan, inplace=True)
print("Missing values after replacing '?' with NaN:")
display(df.isnull().sum()[df.isnull().sum() > 0])

Missing values after replacing '?' with NaN:


,0
workclass,1836
occupation,1843
native.country,583


### 2.3 Handle Missing Values Appropriately

In [ ]:
for col in df.columns[df.isnull().any()]:
    if df[col].dtype == 'object':
        df[col].fillna(df[col].mode()[0], inplace=True)
    else:
        df[col].fillna(df[col].median(), inplace=True)
print("Missing values handled. Remaining nulls:", df.isnull().sum().sum())

Missing values handled. Remaining nulls: 0


/tmp/ipykernel_2377/1598949525.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)


### 2.4 Remove Duplicate Records

In [ ]:
print(f"Duplicates before: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True)
print(f"Duplicates after: {df.duplicated().sum()}")

Duplicates before: 24
Duplicates after: 0


### 2.5 Verify Data Quality After Cleaning

In [ ]:
print("Total missing values after cleaning:", df.isnull().sum().sum())

print(f"Dataset shape after cleaning: {df.shape}")

print("First 5 rows after cleaning:")
display(df.head())

### 2.6 Show Before and After Cleaning Statistics

In [ ]:
print("Summary of Data Cleaning Steps:")
print("1. Missing values (initial): Displayed in section 2.1")
print("2. '?' replaced with NaN: Displayed in section 2.2")
print("3. Missing values handled (imputation): Displayed in section 2.3")
print("4. Duplicate records removed: Displayed in section 2.4")
print("\nDataset is now clean and ready for Feature Engineering.")

## Task 3: Feature Engineering

### 3.1 Separate Numerical and Categorical Columns

In [ ]:
X = df.drop('income', axis=1)
y = df['income']
numerical_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()
print("Features separated.")

Features separated.


### 3.2 Encode Categorical Variables

In [ ]:
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
print(f"Encoded features shape: {X_encoded.shape}")

Encoded features shape: (32537, 97)


### 3.3 Scale Numerical Features

In [ ]:
scaler = StandardScaler()
X_encoded[numerical_cols] = scaler.fit_transform(X_encoded[numerical_cols])
print("Numerical features scaled.")

Numerical features scaled.


### 3.4 Split Dataset into Training and Testing Sets (80:20)

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
print("Data split into 80:20 sets.")

Data split into 80:20 sets.


## Task 4: Model Building

In [ ]:
models = {}
metrics = {'Algorithm': [], 'Accuracy': [], 'Precision': [], 'Recall': [], 'F1 Score': [], 'ROC-AUC': []}

def train_and_evaluate_model(name, model, X_train, y_train, X_test, y_test, metrics_dict, models_dict):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else np.zeros(len(y_test))
    metrics_dict['Algorithm'].append(name)
    metrics_dict['Accuracy'].append(accuracy_score(y_test, y_pred))
    metrics_dict['Precision'].append(precision_score(y_test, y_pred))
    metrics_dict['Recall'].append(recall_score(y_test, y_pred))
    metrics_dict['F1 Score'].append(f1_score(y_test, y_pred))
    metrics_dict['ROC-AUC'].append(roc_auc_score(y_test, y_proba))
    models_dict[name] = {'model': model, 'y_pred': y_pred, 'y_proba': y_proba}

### 4.1 Logistic Regression

In [ ]:
train_and_evaluate_model("Logistic Regression", LogisticRegression(max_iter=1000), X_train, y_train, X_test, y_test, metrics, models)
train_and_evaluate_model("Decision Tree", DecisionTreeClassifier(), X_train, y_train, X_test, y_test, metrics, models)
train_and_evaluate_model("Random Forest", RandomForestClassifier(), X_train, y_train, X_test, y_test, metrics, models)
train_and_evaluate_model("KNN", KNeighborsClassifier(), X_train, y_train, X_test, y_test, metrics, models)
train_and_evaluate_model("SVM", SVC(probability=True), X_train, y_train, X_test, y_test, metrics, models)
print("All models trained and metrics collected.")

All models trained and metrics collected.


## Task 5: Performance Evaluation

In [ ]:
metrics_df = pd.DataFrame(metrics).round(4).sort_values(by='Accuracy', ascending=False)
print("Model Performance Evaluation Table:")
display(metrics_df.style.highlight_max(subset=['Accuracy', 'F1 Score'], color='lightgreen'))

Model Performance Evaluation Table:


,Algorithm,Accuracy,Precision,Recall,F1 Score,ROC-AUC
8,SVM,0.850800,0.750600,0.570200,0.648100,0.894200
0,Logistic Regression,0.850200,0.734400,0.592500,0.655800,0.900200
4,Logistic Regression,0.850200,0.734400,0.592500,0.655800,0.900200
6,Random Forest,0.847300,0.721100,0.596900,0.653200,0.892900
2,Random Forest,0.845900,0.715200,0.598900,0.651900,0.892200
7,KNN,0.827600,0.658400,0.591200,0.623000,0.852900
3,KNN,0.827600,0.658400,0.591200,0.623000,0.852900
1,Decision Tree,0.807200,0.597600,0.611000,0.604200,0.740200
5,Decision Tree,0.805500,0.594100,0.607800,0.600900,0.738000
